In [8]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv


In [9]:
load_dotenv()

True

In [10]:
model = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    temperature = 0.5
)

In [11]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [12]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title} in 100 words only and make sure this is not in .md file format. I want this to be same as plain text with proper headings and all'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [13]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [14]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow = graph.compile()

In [16]:
initial_state = {'title': 'LangGraph: Tool for building LLM workflow using State mecanism'}

final_state = workflow.invoke(initial_state)

print(final_state['outline'])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**LangGraph: State-Driven LLM Workflow Orchestration**

**1. Introduction to LangGraph**
   - Addressing challenges in complex LLM chains and agentic systems.
   - LangGraph's role in building robust, multi-step workflows.

**2. The Central Role of State**
   - Defining "State": A mutable object that carries context and data.
   - How state enables memory, iterative reasoning, and dynamic decision-making across steps.

**3. LangGraph's Core Components**
   - **Nodes**: LLMs, tools, and custom logic functions that read from and modify the shared state.
   - **Edges**: Direct or conditional transitions between nodes, often determined by the current state.

**4. Building Stateful LLM Workflows**
   - Illustrating how state evolves as it passes through different nodes and edges.
   - Example: An agent updating a "research_findings" state through multiple tool calls.

**5. Advantages of State-Based Orchestration**
   - Enables complex reasoning, error handling, and dynamic routing.
   - Fac